# FingerBall Stage 2F — Interactive 3D Hand Skeleton Viewer

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/atstuyuki/FingerBall-Analysis/blob/main/notebooks/FingerBall_v2_Stage2F_3D_Hand_Skeleton_Viewer.ipynb)

MediaPipe Hand Landmarkerの21点2D＋相対Zを利用し、**時間スライダー＋縦横360度回転**を行うビューアです。オフラインHTML、横回転・縦回転MP4、代表角度PNG、ZIPをGoogle Driveに保存します。

**科学的限界：相対Zは実寸3Dではありません。単眼推定による疑似3D表示であり、見えない背面を実測した360度復元ではありません。**

In [ ]:
%pip -q install "plotly>=5.18" numpy pandas matplotlib opencv-python-headless tqdm

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import json,zipfile,math
import numpy as np,pandas as pd,plotly.graph_objects as go
import plotly.io as pio
import matplotlib.pyplot as plt,cv2
from tqdm.auto import tqdm

## 1. Google DriveのCSVを選択・設定

In [ ]:
PROJECT_PATH='/content/drive/MyDrive/母指回内トラッキング'
SOURCE_STAGE='stage2d' # 'stage2d' または 'stage2e'
CSV_PATH='' # 空なら指定Stageから自動選択
VIDEO_STEM='7A5812501DC5F7120DF842D06EE1B74C695459F1'
START_FRAME,END_FRAME=175,210
REPRESENTATIVE_FRAME=198
Z_DEPTH_MULTIPLIER=1.0
ROTATION_FRAMES=72
ROTATION_FPS=24
EXPORT_MP4=True
project=Path(PROJECT_PATH)
if CSV_PATH: csv_path=Path(CSV_PATH)
else:
    root=project/f'FingerBall_v2_{SOURCE_STAGE}_results'
    csv_path=root/VIDEO_STEM/'hand_21_landmarks_long.csv'
    if not csv_path.exists():
        choices=sorted(root.glob('*/hand_21_landmarks_long.csv'),key=lambda p:p.stat().st_mtime,reverse=True)
        if not choices:raise FileNotFoundError('CSVが見つかりません。CSV_PATHを指定してください')
        csv_path=choices[0]
if not csv_path.is_file():raise FileNotFoundError(csv_path)
out_dir=project/'FingerBall_v2_stage2f_results'/csv_path.parent.name
out_dir.mkdir(parents=True,exist_ok=True)
print('CSV:',csv_path,'\nOutput:',out_dir)

## 2. 21点データの欠損処理と手首原点の疑似3D座標

In [ ]:
df=pd.read_csv(csv_path)
req={'frame','landmark_id','x_px','y_px','z_relative','detected'}
if req-set(df.columns):raise ValueError('CSVの必須列がありません: '+str(req-set(df.columns)))
if df.duplicated(['frame','landmark_id']).any():raise ValueError('frame/landmark_idに重複があります')
for key in ['x_px','y_px','z_relative']:df[key]=pd.to_numeric(df[key],errors='coerce')
WIDTH=1206.
if 'x_norm' in df.columns:
    a=df[['x_px','x_norm']].dropna()
    a=a[a.x_norm.abs()>1e-7]
    if len(a):WIDTH=float(np.median(a.x_px/a.x_norm))
if not np.isfinite(WIDTH) or WIDTH<=0:raise ValueError('画像幅を確認してください')
FINGERS={
 'Thumb':[(0,1),(1,2),(2,3),(3,4)],
 'Index':[(0,5),(5,6),(6,7),(7,8)],
 'Middle':[(0,9),(9,10),(10,11),(11,12)],
 'Ring':[(0,13),(13,14),(14,15),(15,16)],
 'Little':[(0,17),(17,18),(18,19),(19,20)]}
COLORS={'Thumb':'#f59e0b','Index':'#06b6d4','Middle':'#d946ef','Ring':'#94a3b8','Little':'#64748b'}
TIPS={'Thumb':4,'Index':8,'Middle':12}
frames=list(range(START_FRAME,END_FRAME+1));coords={}
for f in frames:
    g=df[df.frame.eq(f)&df.landmark_id.between(0,20)].sort_values('landmark_id')
    if len(g)!=21 or g.landmark_id.tolist()!=list(range(21)):
        coords[f]=None;continue
    xyz=g[['x_px','y_px','z_relative']].to_numpy(float)
    if not np.isfinite(xyz).all() or not np.all(g.detected.astype(int).to_numpy()==1):
        coords[f]=None;continue
    xyz=xyz-xyz[0]
    coords[f]=np.column_stack([xyz[:,0]/WIDTH,-xyz[:,1]/WIDTH,-xyz[:,2]*Z_DEPTH_MULTIPLIER])
valid=[f for f in frames if coords[f] is not None]
if not valid:raise RuntimeError('有効な21点フレームがありません')
if REPRESENTATIVE_FRAME not in valid:
    old=REPRESENTATIVE_FRAME;REPRESENTATIVE_FRAME=min(valid,key=lambda f:abs(f-old))
    print('代表フレームを変更:',old,'->',REPRESENTATIVE_FRAME)
R=max(.06,max(float(np.max(np.abs(coords[f]))) for f in valid)*1.2)
print('Valid:',len(valid),'/',len(frames),'representative:',REPRESENTATIVE_FRAME)

## 3. 時間スライダー・マウス360度回転付きPlotlyビューア

In [ ]:
def traces(p):
    result=[]
    for finger,edges in FINGERS.items():
        x=[];y=[];z=[]
        for a,b in edges:
            for k in (a,b):
                if p is None:x.append(None);y.append(None);z.append(None)
                else:x.append(float(p[k,0]));y.append(float(p[k,1]));z.append(float(p[k,2]))
            x.append(None);y.append(None);z.append(None)
        result.append(go.Scatter3d(x=x,y=y,z=z,mode='lines',
            line=dict(color=COLORS[finger],width=9),name=finger))
    for finger,j in TIPS.items():
        v=None if p is None else p[j]
        result.append(go.Scatter3d(x=[] if v is None else [float(v[0])],
            y=[] if v is None else [float(v[1])],z=[] if v is None else [float(v[2])],
            mode='markers',marker=dict(color=COLORS[finger],size=7),name=finger+' tip',showlegend=False))
    return result
fig=go.Figure(data=traces(coords[START_FRAME]),
              frames=[go.Frame(name=str(f),data=traces(coords[f]),
                     layout=go.Layout(title=f'Frame {f}: '+('Valid' if coords[f] is not None else 'Unavailable')))
                      for f in frames])
fig.update_layout(template='plotly_dark',height=730,
  title='FingerBall — AI-estimated pseudo-3D hand skeleton',
  scene=dict(xaxis=dict(title='Image X',range=[-R,R]),yaxis=dict(title='Image Y up',range=[-R,R]),
     zaxis=dict(title='Relative Z (not metric)',range=[-R,R]),
     aspectmode='cube',camera=dict(eye=dict(x=1.6,y=1.2,z=1.2)),dragmode='orbit'),
  uirevision='preserve_camera',
  sliders=[dict(currentvalue=dict(prefix='Frame '),steps=[
    dict(label=str(f),method='animate',args=[[str(f)],
       dict(mode='immediate',frame=dict(duration=0,redraw=True),transition=dict(duration=0))])
    for f in frames])],
  updatemenus=[dict(type='buttons',showactive=False,buttons=[
      dict(label='Play',method='animate',args=[None,dict(frame=dict(duration=300,redraw=True),fromcurrent=True)]),
      dict(label='Pause',method='animate',args=[[None],dict(mode='immediate',frame=dict(duration=0,redraw=True))])])])
fig.show(config=dict(scrollZoom=True,displaylogo=False))